# Assignment 02 | Threat Intelligence Analysis for Data Security

**Guizhou University · Information System Security · Muhammad Asim Iqbal**

Open this notebook in Google Colab. Keep your uploaded CSV files in **/content** and run the cells in order. The default input is **waimai_10k.csv**. You can choose a different corpus in Step 1. A GPU runtime speeds up the three neural models, and the notebook also supports CPU.

The notebook covers the seven parts of Assignment 02. It generates results and a Word report draft from your actual run. After the run, capture the requested Colab screenshots, review the Chinese topics, and add your own interpretation and course feedback before submission. Submit the report with the source notebook. The assignment sheet gives the deadline as **before noon on 30 November 2026**.

| Assignment part | Notebook evidence |
| --- | --- |
| 1. Survey, motivation, and data statistics | Background, corpus inventory, data profile, class and length charts |
| 2. Cleaning, Chinese segmentation, and vectors | Cleaning audit, segmentation example, TF-IDF, padded token IDs and embeddings |
| 3. Machine learning sentiment | Majority baseline, logistic regression, linear SVM, and multinomial Naive Bayes |
| 4. Deep learning sentiment | TextCNN, GRU, and BiLSTM with learning curves and shared test metrics |
| 5. Topic mining | LDA, topic terms and prevalence, held-out perplexity, and a security screening exercise |
| 6. Extension and summary | Error analysis by text length, limitations, and prompts for your own conclusions |
| 7. Course suggestions | Editable prompts for your own feedback and technical questions |

### Part 1 · Concepts and motivation

**Data security** protects confidentiality, integrity, and availability throughout collection, storage, analysis, sharing, and deletion. A practical workflow is to identify sensitive fields, restrict access, document transformations, audit outputs, and control retention.

**Sentiment analysis** predicts the polarity of text. Here the workflow is to clean Chinese reviews, segment them into words, fit a representation on training data, train models, and evaluate on a held-out test set.

**Threat intelligence** involves collecting information, validating and enriching observations, analyzing evidence, and distributing findings for action. In this assignment, sentiment and topic mining illustrate how analysts could triage text for follow-up. Review datasets contain consumer or social opinions. A negative review, security keyword, or LDA topic alone provides no verification of an incident. The security exercise produces aggregate candidate signals for human review.

**Research question.** How well do traditional and neural classifiers identify sentiment in one Chinese corpus, and what recurring themes does unsupervised topic mining reveal?

In [ ]:
# 0. Install the small extra packages. Colab normally supplies PyTorch and scikit-learn.
%pip -q install jieba==0.42.1 python-docx

In [ ]:
# 1. Configuration. Edit these settings before running the other cells.
from pathlib import Path
import os
import hashlib
import importlib.metadata as metadata
import json
import random
import re
import unicodedata
from collections import Counter

import jieba
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split

DATA_ROOT = Path("/content")
DATASET = "waimai"  # options: waimai, chnsenticorp, weibo, shopping
CSV_PATH = None     # example: "/content/online_shopping_dir/online_shopping_10_cats.csv"
TEXT_COLUMN = None  # set only if automatic detection needs correction
LABEL_COLUMN = None
LABEL_OVERRIDES = {}  # explicit mapping for unusual labels, e.g. {"negative": 0, "positive": 1}
DATA_SOURCE = ""  # paste the source URL or course-provided dataset description if known

STUDENT_NAME = "Muhammad Asim Iqbal"
STUDENT_ID = ""
RESEARCH_DIRECTION = ""
SUPERVISOR = ""
MAJOR_AND_CLASS = ""

SEED = 42
MAX_ROWS = 20_000   # stratified cap after cleaning; set None to use all clean rows
EPOCHS = 5
PATIENCE = 2
BATCH_SIZE = 128
MAX_TOKENS = 96
MAX_VOCAB = 15_000
N_TOPICS = 5
LDA_MAX_DOCS = 5_000

# Use a fresh results folder each time to keep runs separate.
from datetime import datetime, timezone
run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
OUT = DATA_ROOT / f"assignment02_{run_id}"
OUT.mkdir(parents=True, exist_ok=False)
FIG = OUT / "figures"
FIG.mkdir()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
plt.style.use("seaborn-v0_8-whitegrid")
print("Runtime:", device, "| Output folder:", OUT)

In [ ]:
# 2. Find the CSVs shown in your Colab Files panel and inspect their schemas.
CORPORA = {
    "waimai": ("waimai_10k.csv",),
    "chnsenticorp": ("ChnSentiCorp.csv",),
    "weibo": ("weibo_senti_100k.csv",),
    "shopping": ("online_shopping_10_cats.csv", "online_shopping.csv"),
}
csvs = sorted(DATA_ROOT.glob("*.csv"))
for folder in ("emotion_nlp_dir", "online_shopping_dir"):
    csvs += sorted((DATA_ROOT / folder).rglob("*.csv"))
csvs = sorted(set(csvs))
print("Discovered CSV files:")
for file in csvs:
    print(" ", file)
if DATASET not in CORPORA and CSV_PATH is None:
    raise ValueError(f"Choose one of {list(CORPORA)} or set CSV_PATH.")
expected = {s.casefold() for s in CORPORA.get(DATASET, ())}
matches = [p for p in csvs if p.name.casefold() in expected]
path = Path(CSV_PATH) if CSV_PATH else (matches[0] if matches else None)
if path is None or not path.is_file():
    raise FileNotFoundError("Upload/extract your CSV into /content or set CSV_PATH to its full path.")
if len(matches) > 1 and not CSV_PATH:
    print("Multiple candidates found. Using", path, "| Set CSV_PATH to choose another.")

def read_csv_flexible(file, **kwargs):
    for encoding in ("utf-8-sig", "gb18030"):
        try:
            return pd.read_csv(file, encoding=encoding, low_memory=False, **kwargs), encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"Cannot decode {file.name} as UTF-8 or GB18030.")

inventory_rows = []
for file in sorted(set(csvs + [path])):
    try:
        preview, enc = read_csv_flexible(file, nrows=3)
        inventory_rows.append({
            "file": str(file.relative_to(DATA_ROOT)) if file.is_relative_to(DATA_ROOT) else file.name,
            "size_kib": round(file.stat().st_size / 1024, 1),
            "encoding": enc,
            "columns": ", ".join(map(str, preview.columns)),
        })
    except (UnicodeError, pd.errors.ParserError, pd.errors.EmptyDataError) as exc:
        inventory_rows.append({"file": file.name, "columns": f"Could not inspect: {type(exc).__name__}"})
inventory = pd.DataFrame(inventory_rows)
display(inventory)
inventory.to_csv(OUT / "corpus_inventory.csv", index=False, encoding="utf-8-sig")

raw, encoding = read_csv_flexible(path)
raw.columns = [str(c).strip() for c in raw.columns]

def choose_column(override, aliases, field):
    if override is not None:
        if override not in raw.columns:
            raise ValueError(f"{field} column {override!r} is absent. Available: {list(raw.columns)}")
        return override
    lookup = {c.casefold(): c for c in raw.columns}
    for alias in aliases:
        if alias.casefold() in lookup:
            return lookup[alias.casefold()]
    raise ValueError(f"Cannot identify {field}. Set its column override. Available: {list(raw.columns)}")

text_col = choose_column(TEXT_COLUMN, ["review", "text", "content", "comment", "sentence", "微博", "评论", "内容"], "text")
label_col = choose_column(LABEL_COLUMN, ["label", "sentiment", "polarity", "target", "标签", "情感"], "label")
if text_col == label_col:
    raise ValueError("The text and label columns must be different.")
print(f"Selected {path.name}: {len(raw):,} rows | {encoding} | text={text_col} | label={label_col}")
display(raw[label_col].value_counts(dropna=False).rename("raw_rows").to_frame())

### Part 2 · Cleaning and Chinese word segmentation

Unicode normalization handles width variants, and jieba splits Chinese text into words. Punctuation-only tokens and identifier placeholders are excluded from model inputs. Negation words such as 不 and 没 remain in the text. Exact duplicate token sequences are removed, and token sequences with conflicting labels are excluded. Pattern masking covers common formats and does not guarantee de-identification. Report charts and exported predictions use counts or row IDs rather than full reviews.

In [ ]:
# 3. Normalize and mask text, segment Chinese, and record the cleaning flow.
URL = re.compile(r"(?:https?://|www\.)[^\s<>\u3000-\u303f\uff00-\uffef\u3400-\u9fff]+", re.I)
EMAIL = re.compile(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", re.ASCII)
HANDLE = re.compile(r"@[\w\u3400-\u9fff-]+")
LONG_NUMBER = re.compile(r"(?<!\d)\d{7,}(?!\d)")
HTML = re.compile(r"<[^>]{1,150}>")
MASK_RULES = {"URL": URL, "Email": EMAIL, "Handle": HANDLE, "Long number": LONG_NUMBER}
LABELS = {
    "0": 0, "0.0": 0, "negative": 0, "neg": 0, "负面": 0, "负向": 0, "差评": 0,
    "1": 1, "1.0": 1, "positive": 1, "pos": 1, "正面": 1, "正向": 1, "好评": 1,
}
LABELS.update({str(k).strip().casefold(): v for k, v in LABEL_OVERRIDES.items()})
if not set(LABELS.values()).issubset({0, 1}):
    raise ValueError("LABEL_OVERRIDES must map to 0 (negative) or 1 (positive).")

def normalize_text(value):
    value = unicodedata.normalize("NFKC", str(value)).lower()
    return HTML.sub(" ", value)

def clean_text(value):
    for pattern in MASK_RULES.values():
        value = pattern.sub(" ", value)
    return " ".join(value.split())

def tokenize(value):
    return [
        token.strip() for token in jieba.lcut(value, HMM=True)
        if token.strip() and any(ch.isalnum() for ch in token)
    ]

normalized = raw[text_col].fillna("").map(normalize_text)
mask_stats = pd.DataFrame([
    {"pattern": name, "raw_documents_with_match": int(normalized.map(lambda s: bool(pattern.search(s))).sum())}
    for name, pattern in MASK_RULES.items()
])
mask_stats.to_csv(OUT / "privacy_pattern_audit.csv", index=False)
mapped = raw[label_col].astype("string").str.strip().str.casefold().map(LABELS)
if not mapped.notna().any():
    raise ValueError("No binary labels recognized. Check the raw label counts and set LABEL_OVERRIDES.")
data = pd.DataFrame({"source_row": np.arange(len(raw)), "label": mapped, "text": normalized.map(clean_text)})
audit_rows = [{"stage": "Raw input", "retained": len(data), "removed_at_stage": 0}]

def record_stage(stage, size):
    audit_rows.append({"stage": stage, "retained": size, "removed_at_stage": audit_rows[-1]["retained"] - size})

data = data.loc[data["label"].notna()].copy()
data["label"] = data["label"].astype(int)
record_stage("Valid binary label", len(data))
data = data.loc[data["text"].str.len() > 0].copy()
record_stage("Nonempty after normalization and masking", len(data))
data["tokens"] = data["text"].map(tokenize)
data = data.loc[data["tokens"].map(len) > 0].copy()
record_stage("At least one retained token", len(data))
data["segmented"] = data["tokens"].map(" ".join)
conflicts = data.groupby("segmented", sort=False)["label"].transform("nunique") > 1
data = data.loc[~conflicts].copy()
record_stage("Conflicting token sequences excluded", len(data))
data = data.drop_duplicates("segmented", keep="first").copy()
record_stage("Exact token duplicates removed", len(data))
data["word_count"] = data["tokens"].map(len)
data["char_count"] = data["text"].str.len()

if data["label"].nunique() != 2 or data["label"].value_counts().min() < 10:
    raise ValueError("Each class needs at least 10 distinct labeled reviews after cleaning.")
clean_count = len(data)
if MAX_ROWS is not None and len(data) > MAX_ROWS:
    if MAX_ROWS < 30:
        raise ValueError("Set MAX_ROWS to at least 30 or None.")
    data, _ = train_test_split(data, train_size=MAX_ROWS, stratify=data["label"], random_state=SEED)
record_stage("Selected modeling corpus", len(data))
data = data.reset_index(drop=True)
if data["label"].nunique() != 2 or data["label"].value_counts().min() < 10:
    raise ValueError("The modeling cap removed too many minority examples. Increase MAX_ROWS.")

audit = pd.DataFrame(audit_rows)
audit.to_csv(OUT / "cleaning_audit.csv", index=False)
display(audit, mask_stats)
example = "送餐很快，但是包装不太好。联系方式：test@example.com"
print("Synthetic demonstration:", example)
print("After cleaning:", clean_text(normalize_text(example)))
print("Jieba tokens:", tokenize(clean_text(normalize_text(example))))

In [ ]:
# 4. Explore distributions and freeze a common stratified train/validation/test split.
def save_chart(fig, filename):
    fig.tight_layout()
    fig.savefig(FIG / filename, dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

profile = data.groupby("label").agg(
    rows=("source_row", "size"),
    mean_characters=("char_count", "mean"),
    median_tokens=("word_count", "median"),
    p95_tokens=("word_count", lambda s: s.quantile(.95)),
).reset_index()
profile.to_csv(OUT / "dataset_profile.csv", index=False)
display(profile)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
counts = data["label"].value_counts().reindex([0, 1])
ax[0].bar(["Negative (0)", "Positive (1)"], counts, color=["#c25a41", "#267c77"])
ax[0].set(title="Sentiment class distribution", ylabel="Reviews")
upper = max(5, int(data["word_count"].quantile(.99)))
for label, name, color in [(0, "Negative", "#c25a41"), (1, "Positive", "#267c77")]:
    lengths = data.loc[data.label.eq(label), "word_count"].clip(upper=upper)
    ax[1].hist(lengths, bins=np.linspace(0, upper + 1, 22), alpha=.55, color=color, label=name)
ax[1].set(title="Token length by sentiment", xlabel=f"Tokens (right tail capped at {upper})", ylabel="Reviews")
ax[1].legend()
save_chart(fig, "01_dataset_profile.png")

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(audit["stage"], audit["retained"], color="#306a92")
ax.invert_yaxis()
ax.set(title="Records retained at each cleaning step", xlabel="Rows")
save_chart(fig, "02_cleaning_flow.png")

train_idx, hold_idx = train_test_split(
    np.arange(len(data)), test_size=.30, stratify=data["label"], random_state=SEED
)
val_idx, test_idx = train_test_split(
    hold_idx, test_size=.50, stratify=data.iloc[hold_idx]["label"], random_state=SEED
)
sets = {"train": data.iloc[train_idx].copy(), "validation": data.iloc[val_idx].copy(),
        "test": data.iloc[test_idx].copy()}
assert sum(len(part) for part in sets.values()) == len(data)
assert len(set.union(*(set(part["source_row"]) for part in sets.values()))) == len(data)
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    assert set(sets[a]["segmented"]).isdisjoint(sets[b]["segmented"]), (a, b)
for name, part in sets.items():
    if part["label"].nunique() != 2:
        raise ValueError(f"{name} lacks a class. Use more data or adjust the modeling cap.")
split_table = pd.concat([part[["source_row", "label", "word_count"]].assign(split=name)
                         for name, part in sets.items()], ignore_index=True)
split_table.to_csv(OUT / "split_manifest.csv", index=False)
split_counts = pd.crosstab(split_table["split"], split_table["label"]).reset_index()
split_counts.to_csv(OUT / "split_counts.csv", index=False)
display(split_counts)
print("Audit passed: no identical token sequences cross partitions.")

In [ ]:
# 5. TF-IDF for classic models and train-only token IDs for neural models.
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    tokenizer=str.split, token_pattern=None, lowercase=False, ngram_range=(1, 2),
    min_df=2, max_features=40_000, sublinear_tf=True, dtype=np.float32
)
X_train = tfidf.fit_transform(sets["train"]["segmented"])
X_val = tfidf.transform(sets["validation"]["segmented"])
X_test = tfidf.transform(sets["test"]["segmented"])
y = {name: part["label"].to_numpy(dtype=np.int64) for name, part in sets.items()}

token_freq = Counter(token for doc in sets["train"]["tokens"] for token in doc)
vocab = {token: i + 2 for i, (token, _) in enumerate(token_freq.most_common(MAX_VOCAB - 2))}
PAD_ID, UNK_ID = 0, 1
vocab_size = len(vocab) + 2

def encode_batch(documents):
    sequences = np.full((len(documents), MAX_TOKENS), PAD_ID, dtype=np.int64)
    lengths = np.zeros(len(documents), dtype=np.int64)
    for i, tokens in enumerate(documents):
        ids = [vocab.get(token, UNK_ID) for token in tokens[:MAX_TOKENS]]
        sequences[i, :len(ids)] = ids
        lengths[i] = len(ids)
    return sequences, lengths

encoded = {name: encode_batch(part["tokens"]) for name, part in sets.items()}
repr_stats = pd.DataFrame([
    {"split": name, "documents": len(part), "tfidf_features": X_train.shape[1],
     "tokens_truncated_docs": int(part["word_count"].gt(MAX_TOKENS).sum()),
     "unknown_token_rate": round(float((encoded[name][0] == UNK_ID).sum() / encoded[name][1].sum()), 4)}
    for name, part in sets.items()
])
repr_stats.to_csv(OUT / "representation_stats.csv", index=False)
display(repr_stats)
print("TF-IDF matrix shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Neural sequence matrix:", encoded["train"][0].shape, "| vocabulary:", vocab_size)
print("First training sequence IDs (no text):", encoded["train"][0][0, :12])

### Part 3 · Traditional machine learning

The **majority baseline** predicts the most frequent training class. **Logistic regression** estimates a linear decision boundary from TF-IDF features. A **linear SVM** learns a margin and produces decision scores. **Multinomial Naive Bayes** uses nonnegative term weights to estimate class evidence under a conditional independence assumption. Use accuracy alongside macro F1, which gives equal weight to the two classes.

In [ ]:
# 6. Train classic models. Keep the test set for the final comparison.
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score, roc_auc_score)
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

results = []
predictions = {}
test_labels = y["test"]

def log_test_result(name, family, pred, scores):
    predictions[name] = (np.asarray(pred), np.asarray(scores))
    results.append({
        "model": name, "family": family, "accuracy": accuracy_score(test_labels, pred),
        "balanced_accuracy": balanced_accuracy_score(test_labels, pred),
        "macro_f1": f1_score(test_labels, pred, average="macro", zero_division=0),
        "positive_f1": f1_score(test_labels, pred, zero_division=0),
        "roc_auc": roc_auc_score(test_labels, scores),
        "average_precision": average_precision_score(test_labels, scores),
    })

classic = {
    "Majority": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
    "Linear SVM": LinearSVC(class_weight="balanced", max_iter=5000, random_state=SEED),
    "Multinomial NB": MultinomialNB(alpha=1.0),
}
validation_rows = []
for name, model in classic.items():
    model.fit(X_train, y["train"])
    val_f1 = f1_score(y["validation"], model.predict(X_val), average="macro", zero_division=0)
    validation_rows.append({"model": name, "validation_macro_f1": val_f1})
    pred = model.predict(X_test)
    scores = model.decision_function(X_test) if hasattr(model, "decision_function") else model.predict_proba(X_test)[:, 1]
    log_test_result(name, "Baseline" if name == "Majority" else "Traditional ML", pred, scores)

display(pd.DataFrame(results).round(4))

### Part 4 · Three deep learning architectures

1. **TextCNN** uses convolutions over adjacent words and max pooling to capture local phrases.
2. **GRU** reads a word sequence forward with gates that control its hidden state.
3. **BiLSTM** reads the sequence in both directions and combines the final states.

All three learn embeddings from this dataset. The training loss uses weights based on training class frequencies. Validation macro F1 selects each checkpoint. The final test set is shared with the classical models. These are single-seed experiments, so small score differences need further verification.

In [ ]:
# 7. Define neural models and masked/padded sequence handling.
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

def make_loader(name, shuffle=False):
    seq, length = encoded[name]
    ds = TensorDataset(torch.from_numpy(seq), torch.from_numpy(length), torch.from_numpy(y[name]))
    return DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
        generator=torch.Generator().manual_seed(SEED) if shuffle else None,
        pin_memory=device.type == "cuda"
    )

class TextCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=PAD_ID)
        self.convs = nn.ModuleList(nn.Conv1d(64, 64, width, padding=width // 2) for width in (3, 5, 7))
        self.head = nn.Sequential(nn.Dropout(.30), nn.Linear(192, 2))

    def forward(self, seq, lengths):
        x = self.embedding(seq).transpose(1, 2)
        valid = torch.arange(seq.shape[1], device=seq.device)[None, :] < lengths.to(seq.device)[:, None]
        features = [torch.relu(conv(x)).masked_fill(~valid[:, None, :], -torch.inf).amax(dim=2)
                    for conv in self.convs]
        return self.head(torch.cat(features, dim=1))

class RecurrentClassifier(nn.Module):
    def __init__(self, kind):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=PAD_ID)
        self.kind = kind
        if kind == "GRU":
            self.rnn = nn.GRU(64, 64, batch_first=True)
        elif kind == "BiLSTM":
            self.rnn = nn.LSTM(64, 64, batch_first=True, bidirectional=True)
        else:
            raise ValueError(f"Unknown recurrent architecture: {kind}")
        self.head = nn.Sequential(nn.Dropout(.30), nn.Linear(64 if kind == "GRU" else 128, 2))

    def forward(self, seq, lengths):
        packed = pack_padded_sequence(self.embedding(seq), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, state = self.rnn(packed)
        if self.kind == "GRU":
            features = state[-1]
        else:
            hidden, _ = state
            features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.head(features)

train_class_counts = np.bincount(y["train"], minlength=2)
weights = torch.tensor(len(y["train"]) / (2.0 * train_class_counts), dtype=torch.float32, device=device)
loss_fn = nn.CrossEntropyLoss(weight=weights)

In [ ]:
# 8. Fit each architecture and select its checkpoint on validation macro F1.
import time

@torch.inference_mode()
def neural_predictions(model, loader):
    model.eval()
    scores = []
    for seq, lengths, _ in loader:
        logits = model(seq.to(device), lengths)
        scores.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    scores = np.concatenate(scores)
    return (scores >= .5).astype(int), scores

def train_neural(name, make_model):
    torch.manual_seed(SEED)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(SEED)
    model = make_model().to(device)
    train_loader = make_loader("train", shuffle=True)
    val_loader = make_loader("validation")
    optimizer = torch.optim.AdamW(model.parameters(), lr=.001)
    history, best_state = [], None
    best_f1, best_epoch, stale = -1., 0, 0
    for epoch in range(1, EPOCHS + 1):
        start = time.perf_counter()
        model.train()
        total_loss, total_weight = 0., 0.
        for seq, lengths, labels in train_loader:
            seq, labels = seq.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(seq, lengths), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            batch_weight = weights[labels].sum().item()
            total_loss += loss.item() * batch_weight
            total_weight += batch_weight
        val_pred, _ = neural_predictions(model, val_loader)
        val_f1 = f1_score(y["validation"], val_pred, average="macro", zero_division=0)
        history.append({"model": name, "epoch": epoch, "training_loss": total_loss / total_weight,
                        "validation_macro_f1": val_f1, "seconds": time.perf_counter() - start})
        print(f"{name:8s} epoch {epoch:02d} | loss {history[-1]['training_loss']:.4f} | val macro F1 {val_f1:.4f}")
        if val_f1 > best_f1 + 1e-6:
            best_f1, best_epoch, stale = val_f1, epoch, 0
            best_state = {key: tensor.detach().cpu().clone() for key, tensor in model.state_dict().items()}
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    model.load_state_dict(best_state)
    pred, scores = neural_predictions(model, make_loader("test"))
    return history, best_f1, best_epoch, pred, scores

if EPOCHS < 1 or PATIENCE < 1:
    raise ValueError("EPOCHS and PATIENCE must be at least 1.")
histories = []
# Re-running this cell replaces neural results and retains classic results.
results = [row for row in results if row["family"] != "Deep learning"]
validation_rows = [row for row in validation_rows if row["model"] not in {"TextCNN", "GRU", "BiLSTM"}]
for name, factory in [
    ("TextCNN", TextCNN),
    ("GRU", lambda: RecurrentClassifier("GRU")),
    ("BiLSTM", lambda: RecurrentClassifier("BiLSTM")),
]:
    hist, val_f1, epoch, pred, scores = train_neural(name, factory)
    histories.extend(hist)
    validation_rows.append({"model": name, "validation_macro_f1": val_f1, "best_epoch": epoch})
    log_test_result(name, "Deep learning", pred, scores)

history_table = pd.DataFrame(histories)
history_table.to_csv(OUT / "neural_training_history.csv", index=False)
validation_table = pd.DataFrame(validation_rows)
validation_table.to_csv(OUT / "validation_scores.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, g in history_table.groupby("model", sort=False):
    ax[0].plot(g.epoch, g.training_loss, marker="o", label=name)
    ax[1].plot(g.epoch, g.validation_macro_f1, marker="o", label=name)
ax[0].set(title="Training loss", xlabel="Epoch", ylabel="Weighted cross-entropy")
ax[1].set(title="Validation performance", xlabel="Epoch", ylabel="Macro F1", ylim=(0, 1))
ax[0].legend()
ax[1].legend()
save_chart(fig, "03_neural_learning_curves.png")

In [ ]:
# 9. Shared test evaluation, confusion matrices, curves, and error analysis.
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay

result_table = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
result_table.to_csv(OUT / "test_model_comparison.csv", index=False)
display(result_table.round(4))

# Choose one model for detailed analysis by validation only, excluding the dummy.
selected = validation_table.loc[validation_table.model.ne("Majority")].sort_values(
    "validation_macro_f1", ascending=False, kind="stable"
).iloc[0]["model"]
pred, scores = predictions[selected]
print("Selected by validation macro F1:", selected)
report_metrics = pd.DataFrame(classification_report(
    test_labels, pred, labels=[0, 1], target_names=["negative", "positive"],
    output_dict=True, zero_division=0
)).T
display(report_metrics.round(4))
report_metrics.to_csv(OUT / "selected_model_classification_report.csv")

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_scores = result_table.set_index("model")[["macro_f1", "roc_auc", "average_precision"]]
plot_scores.plot.bar(ax=ax, color=["#267c77", "#375b89", "#c88938"], width=.8)
ax.set(title="Model comparison on the same test set", ylabel="Score", ylim=(0, 1))
ax.tick_params(axis="x", labelrotation=22)
save_chart(fig, "04_model_comparison.png")

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
model_names = [name for name in predictions if name != "Majority"]
for name, ax in zip(model_names, axes.flat):
    ConfusionMatrixDisplay.from_predictions(test_labels, predictions[name][0], labels=[0, 1],
                                           display_labels=["Neg", "Pos"], colorbar=False, ax=ax)
    ax.set_title(name)
save_chart(fig, "05_confusion_matrices.png")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for name in model_names:
    RocCurveDisplay.from_predictions(test_labels, predictions[name][1], name=name, ax=ax[0])
    PrecisionRecallDisplay.from_predictions(test_labels, predictions[name][1], name=name, ax=ax[1])
ax[0].plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
ax[1].axhline(test_labels.mean(), linestyle="--", color="gray", linewidth=1, label="Positive prevalence")
ax[0].set_title("ROC curves on the test set")
ax[1].set_title("Precision-recall curves on the test set")
for axis in ax:
    axis.legend(fontsize=7, loc="lower right")
save_chart(fig, "06_roc_precision_recall.png")

# Export row identifiers and predictions, without exporting review text.
all_pred = []
for name, (predicted, score) in predictions.items():
    all_pred.append(pd.DataFrame({
        "source_row": sets["test"]["source_row"].to_numpy(), "model": name,
        "label": test_labels, "prediction": predicted, "score_positive": score,
    }))
pd.concat(all_pred, ignore_index=True).to_csv(OUT / "test_predictions.csv", index=False)
errors = sets["test"][["source_row", "label", "word_count"]].copy()
errors["prediction"] = pred
errors["wrong"] = errors["label"] != errors["prediction"]
errors["length_band"] = pd.cut(errors["word_count"], [0, 10, 30, np.inf], labels=["1–10", "11–30", "31+"])
error_summary = errors.groupby("length_band", observed=True).agg(
    count=("wrong", "size"), errors=("wrong", "sum"), error_rate=("wrong", "mean")
).reset_index()
error_summary.to_csv(OUT / "error_analysis_by_length.csv", index=False)
display(error_summary.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(error_summary["length_band"].astype(str), error_summary["error_rate"], color="#b35f44")
for i, row in enumerate(error_summary.itertuples()):
    ax.text(i, row.error_rate + .025, f"n={row.count}", ha="center", fontsize=9)
ax.set(title=f"Test error by review length | {selected}", xlabel="Token length band",
       ylabel="Error rate", ylim=(0, 1.10))
save_chart(fig, "07_error_by_length.png")

### Part 5 · Topic mining and security screening

Latent Dirichlet Allocation (LDA) models each document as a mixture of topics and each topic as a distribution over words. It uses **word counts**, whereas the sentiment baselines above use TF-IDF. Here we fit on training reviews and estimate topic proportions for held-out reviews. Topic IDs have no meaning until you read their top words. Report held-out perplexity with its evaluation sample size and explain which topics seem coherent or mixed. Perplexity alone does not establish topic quality or security relevance.

In [ ]:
# 10. LDA on Chinese tokens, plus aggregate screening for selected security terms.
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

lda_train = sets["train"]
if len(lda_train) > LDA_MAX_DOCS:
    lda_train, _ = train_test_split(lda_train, train_size=LDA_MAX_DOCS,
                                    stratify=lda_train["label"], random_state=SEED)
topic_stop = ["的", "了", "是", "我", "也", "就", "都", "在", "和", "有", "又", "很", "还"]
count_vectorizer = CountVectorizer(
    tokenizer=str.split, token_pattern=None, lowercase=False, min_df=2,
    max_df=.95, max_features=5000, stop_words=topic_stop
)
topic_train_matrix = count_vectorizer.fit_transform(lda_train["segmented"]).tocsr()
topic_test_matrix = count_vectorizer.transform(sets["test"]["segmented"]).tocsr()
train_nonzero = np.diff(topic_train_matrix.indptr) > 0
test_nonzero = np.diff(topic_test_matrix.indptr) > 0
if train_nonzero.sum() < N_TOPICS or not test_nonzero.any():
    raise ValueError("Too little retained vocabulary for LDA. Increase data or reduce N_TOPICS/min_df.")
lda = LatentDirichletAllocation(n_components=N_TOPICS, max_iter=12, learning_method="batch", random_state=SEED)
lda.fit(topic_train_matrix[train_nonzero])
topic_probs = lda.transform(topic_test_matrix[test_nonzero])
dominant = topic_probs.argmax(axis=1)
topic_terms = count_vectorizer.get_feature_names_out()
topic_table = pd.DataFrame([
    {"topic": f"Topic {i + 1}",
     "top_terms": " · ".join(topic_terms[np.argsort(weights)[-10:][::-1]]),
     "heldout_mean_weight": float(topic_probs[:, i].mean()),
     "heldout_dominant_count": int((dominant == i).sum())}
    for i, weights in enumerate(lda.components_)
])
lda_perplexity = float(lda.perplexity(topic_test_matrix[test_nonzero]))
topic_table.to_csv(OUT / "lda_topics.csv", index=False, encoding="utf-8-sig")
print(f"Held-out perplexity: {lda_perplexity:.2f} on {int(test_nonzero.sum())}/{len(test_nonzero)} test reviews")
display(topic_table.round(4))

topic_labels = sets["test"]["label"].to_numpy()[test_nonzero]
topic_group = pd.DataFrame(topic_probs, columns=topic_table["topic"].tolist()).assign(label=topic_labels)
topic_group_means = topic_group.groupby("label").mean().reindex([0, 1])
topic_group_means.to_csv(OUT / "lda_topic_weights_by_sentiment.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(topic_table["topic"], topic_table["heldout_mean_weight"], color="#267c77")
ax[0].set(title="LDA topic prevalence on held-out reviews", ylabel="Mean topic weight")
ax[0].tick_params(axis="x", labelrotation=30)
topic_group_means.T.rename(columns={0: "Negative", 1: "Positive"}).plot.bar(
    ax=ax[1], color=["#c25a41", "#306a92"]
)
ax[1].set(title="Topic mixtures by observed sentiment", ylabel="Mean topic weight", xlabel="")
ax[1].tick_params(axis="x", labelrotation=30)
save_chart(fig, "08_lda_topics.png")

# Optional extension: candidate triage counts. Literal matches have no incident ground truth.
keyword_groups = {
    "Privacy and disclosure": ["隐私", "泄露", "个人信息"],
    "Fraud and phishing": ["诈骗", "钓鱼", "欺诈"],
    "Account compromise": ["盗号", "账号被盗", "账户被盗"],
    "Malware and vulnerabilities": ["恶意软件", "木马", "漏洞"],
}
screen = pd.DataFrame([
    {"category": category, "terms": " · ".join(terms),
     "documents_flagged": int(data["text"].str.contains("|".join(map(re.escape, terms)), regex=True).sum())}
    for category, terms in keyword_groups.items()
])
screen.to_csv(OUT / "security_keyword_screen.csv", index=False, encoding="utf-8-sig")
display(screen)
print("Counts can overlap across categories. Zero matches and false positives are possible.")

### Parts 6 and 7 · Write your conclusions and course suggestions

- Compare the validation-selected model with the majority baseline and one model from the other family. Cite actual scores. Discuss whether accuracy and macro F1 tell a similar story.
- Review three misclassifications locally using the row IDs. Discuss possible negation, mixed sentiment, short text, or label noise. Include only redacted examples if your instructor permits them.
- Give descriptive names to at least two LDA topics after reading their terms. Note whether they look like ordinary consumer complaints, possible security-related candidates, or mixed topics. If there are no convincing security-related themes, report that result.
- Discuss dataset domain, single-seed results, class distribution, and incomplete masking. Describe how encryption, access control, and provenance checks would apply to a real security corpus. An optional AI for Science connection is to compare these same controls with the handling of sensitive biomedical text.
- Give one specific suggestion on teaching format, one on course content, and one question for the instructor.

**Required screenshots.** Capture your own Colab outputs after the run. Suggested evidence is Step 3/5 (segmentation and matrix shapes), Step 6 (classic training/results), Step 8/9 (three neural models, logs and evaluation), and Step 10 (topic table and chart). The generated figures can be used alongside the screenshots.

In [ ]:
# 11. Create an editable Word report draft from actual outputs.
from docx import Document
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

def sha256_file(file):
    digest = hashlib.sha256()
    with file.open("rb") as src:
        for block in iter(lambda: src.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

run_metadata = {
    "dataset": path.name, "source": DATA_SOURCE or None, "sha256": sha256_file(path),
    "encoding": encoding, "text_column": text_col,
    "label_column": label_col, "source_row_definition": "zero-based data row position after CSV parsing",
    "raw_rows": len(raw), "clean_rows_before_cap": clean_count, "modeled_rows": len(data),
    "seed": SEED, "max_rows": MAX_ROWS, "epochs": EPOCHS, "patience": PATIENCE,
    "batch_size": BATCH_SIZE, "max_tokens": MAX_TOKENS, "vocabulary_size": vocab_size,
    "lda_topics": N_TOPICS, "lda_max_docs": LDA_MAX_DOCS, "lda_train_documents_used": int(train_nonzero.sum()),
    "lda_test_documents_used": int(test_nonzero.sum()), "lda_heldout_perplexity": lda_perplexity,
    "validation_selected_model": selected, "device": str(device),
    "versions": {p: package_version(p) for p in ["pandas", "numpy", "jieba", "scikit-learn", "torch", "matplotlib", "python-docx"]},
}
(OUT / "run_metadata.json").write_text(json.dumps(run_metadata, ensure_ascii=False, indent=2), encoding="utf-8")

doc = Document()
section = doc.sections[0]
section.top_margin = Inches(.75)
section.bottom_margin = Inches(.75)
doc.styles["Normal"].font.name = "Calibri"
doc.styles["Normal"].font.size = Pt(11)
normal_fonts = doc.styles["Normal"].element.get_or_add_rPr().get_or_add_rFonts()
normal_fonts.set(qn("w:eastAsia"), "Microsoft YaHei")
normal_fonts.attrib.pop(qn("w:eastAsiaTheme"), None)
for name in ["Title", "Heading 1", "Heading 2"]:
    doc.styles[name].font.color.rgb = RGBColor.from_string("173B49")
for style in doc.styles:
    for border in list(style.element.iter(qn("w:pBdr"))):
        border.getparent().remove(border)
doc.core_properties.author = STUDENT_NAME

def add_table(frame):
    table = doc.add_table(rows=1, cols=len(frame.columns))
    table.style = "Light Shading Accent 1"
    for cell, value in zip(table.rows[0].cells, frame.columns):
        cell.text = str(value)
    for values in frame.itertuples(index=False, name=None):
        for cell, value in zip(table.add_row().cells, values):
            cell.text = f"{value:.3f}" if isinstance(value, (float, np.floating)) else str(value)

def add_figure(file, caption):
    doc.add_picture(str(FIG / file), width=Inches(5.8))
    doc.paragraphs[-1].paragraph_format.keep_with_next = True
    doc.add_paragraph(caption, style="Caption")

def write_here(text):
    paragraph = doc.add_paragraph()
    run = paragraph.add_run(f"[{text}]")
    run.italic = True
    run.font.color.rgb = RGBColor.from_string("346A87")

def code_excerpt(text):
    para = doc.add_paragraph()
    run = para.add_run(text)
    run.font.name = "Consolas"
    run.font.size = Pt(9)

doc.add_paragraph("Guizhou University  |  State Key Laboratory of Public Big Data")
doc.add_heading("Threat Intelligence Analysis for Data Security", 0)
doc.add_paragraph("Information System Security  |  Assignment 02  |  November 2026")
for label, value in [
    ("Student name", STUDENT_NAME), ("Student ID", STUDENT_ID),
    ("Research direction", RESEARCH_DIRECTION), ("Supervisor and title", SUPERVISOR),
    ("Major and class", MAJOR_AND_CLASS), ("Course instructor", "Yang Xiuzhang"),
]:
    doc.add_paragraph(f"{label}  {value or '[Fill in]'}")

doc.add_heading("1 Survey and dataset", level=1)
doc.add_paragraph(
    "Data security protects confidentiality, integrity, and availability from collection through deletion. "
    "Sentiment analysis classifies text polarity after cleaning and representation. Threat intelligence "
    "collects, validates, analyzes, and shares evidence about potential threats. This experiment uses "
    "review text to study how sentiment and topic mining can support screening for human review."
)
doc.add_paragraph(f"The selected file was {path.name}. It contained {len(raw):,} input rows and "
                  f"{clean_count:,} retained rows before capping. The modeled corpus contained {len(data):,} reviews.")
if DATA_SOURCE:
    doc.add_paragraph(f"Dataset provenance: {DATA_SOURCE}")
add_table(profile[["label", "rows", "median_tokens", "p95_tokens"]])
add_figure("01_dataset_profile.png", "Figure 1. Class counts and segmented review length.")
write_here("Add the dataset provenance, domain, collection context if known, and your research motivation.")

doc.add_heading("2 Data cleaning and representation", level=1)
doc.add_paragraph(
    "The pipeline normalizes Unicode, removes common identifier patterns, segments Chinese text with jieba, "
    "and filters invalid labels, empty reviews, conflicting duplicates, and exact token duplicates. "
    "A stratified 70/15/15 split is shared by all classifiers. TF-IDF and the neural vocabulary are fit on training data."
)
add_table(audit)
add_table(split_counts.rename(columns={0: "Negative", 1: "Positive"}))
add_figure("02_cleaning_flow.png", "Figure 2. Retained records at each preprocessing stage.")
code_excerpt('tokens = tokenize(clean_text(normalize_text(example)))\nX_train = tfidf.fit_transform(sets["train"]["segmented"])\nX_test = tfidf.transform(sets["test"]["segmented"])')
write_here("Insert your preprocessing screenshot showing jieba tokens and vector shapes from Steps 3 and 5.")

doc.add_heading("3 Traditional machine learning", level=1)
doc.add_paragraph("Logistic regression estimates a linear decision boundary, the SVM maximizes a margin, "
                  "and multinomial Naive Bayes models class evidence with nonnegative features. "
                  "The majority classifier provides a reference point.")
add_table(result_table.loc[result_table.family.ne("Deep learning"),
                           ["model", "accuracy", "macro_f1", "roc_auc", "average_precision"]])
code_excerpt('model.fit(X_train, y["train"])\npred = model.predict(X_test)')
write_here("Insert your Step 6 screenshot and compare the traditional models using actual results.")

doc.add_heading("4 Deep learning and evaluation", level=1)
doc.add_paragraph("TextCNN pools local convolutional features, GRU encodes text in forward order, and BiLSTM "
                  "combines forward and backward states. Validation macro F1 selects checkpoints.")
add_table(result_table.loc[result_table.family.eq("Deep learning"),
                           ["model", "accuracy", "macro_f1", "roc_auc", "average_precision"]])
code_excerpt("loss = loss_fn(model(seq, lengths), labels)\nloss.backward()\noptimizer.step()")
add_figure("03_neural_learning_curves.png", "Figure 3. Training loss and validation macro F1.")
add_figure("04_model_comparison.png", "Figure 4. Comparison on the common test split.")
add_figure("05_confusion_matrices.png", "Figure 5. Test confusion matrices for the six trained classifiers.")
add_figure("06_roc_precision_recall.png", "Figure 6. Test ROC and precision–recall curves.")
doc.add_paragraph(f"Validation-selected model for detailed error analysis: {selected}.")
write_here("Insert your neural training screenshot. Explain one learning curve and compare class errors.")

doc.add_heading("5 Topic mining and threat intelligence screening", level=1)
doc.add_paragraph(f"LDA used word counts from {int(train_nonzero.sum()):,} training reviews. "
                  f"Held-out perplexity was {lda_perplexity:.2f} on {int(test_nonzero.sum()):,} test reviews "
                  "with retained vocabulary. Keyword counts indicate candidate text matches for follow-up.")
add_table(topic_table[["topic", "top_terms", "heldout_mean_weight"]].rename(columns={
    "topic": "Topic", "top_terms": "Top Chinese terms", "heldout_mean_weight": "Mean test weight"
}))
add_figure("08_lda_topics.png", "Figure 7. Topic prevalence and mean topic weights by sentiment.")
add_table(screen[["category", "documents_flagged"]])
code_excerpt("lda.fit(topic_train_matrix[train_nonzero])\ntopic_probs = lda.transform(topic_test_matrix[test_nonzero])")
write_here("Insert your LDA screenshot, name at least two topics from their terms, and assess security relevance.")

doc.add_heading("6 Extension and summary", level=1)
add_figure("07_error_by_length.png", "Figure 8. Test error rates by token length for the validation-selected model.")
write_here("Write your conclusions, limitations, problems encountered, and an optional research extension.")
doc.add_heading("7 Course suggestions", level=1)
write_here("Give a teaching-format suggestion, a remaining-content suggestion, and a technical question.")
doc.add_heading("Technical references", level=1)
for ref in [
    "jieba Chinese segmentation documentation. https://github.com/fxsjy/jieba",
    "Scikit-learn LDA documentation. https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html",
    "Scikit-learn model evaluation. https://scikit-learn.org/stable/modules/model_evaluation.html",
    "PyTorch neural network modules. https://docs.pytorch.org/docs/stable/nn.html",
]:
    doc.add_paragraph(ref)
report_file = OUT / "Assignment_02_Report_Draft.docx"
doc.save(report_file)
print("Created:", report_file)

In [ ]:
# 12. Package report, figures, metrics, audits and settings. Download the source .ipynb separately.
from zipfile import ZipFile, ZIP_DEFLATED

readme = """Assignment 02 result bundle
1. Open Assignment_02_Report_Draft.docx and fill in the marked prompts and cover details.
2. Add your Colab experiment screenshots for preprocessing, traditional ML, neural training, and LDA.
3. Review the generated tables, charts, and data provenance before submission.
4. Download your executed notebook through File > Download > Download .ipynb and submit it with the report.
5. If a .py source file is also requested, use File > Download > Download .py.
The results bundle contains no full review text. Keep your source data and Colab outputs access controlled.
"""
(OUT / "SUBMISSION_CHECKLIST.txt").write_text(readme, encoding="utf-8")
zip_path = OUT.parent / f"{OUT.name}_results.zip"
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for file in sorted(OUT.rglob("*")):
        if file.is_file():
            archive.write(file, file.relative_to(OUT))
print(f"Result bundle: {zip_path}")
print("Download from the Colab Files sidebar, or uncomment the two lines below:")
# from google.colab import files
# files.download(str(zip_path))

### Before you submit

1. Fill in your student ID, research direction, supervisor, and class in the Word file.
2. Add the required screenshots from your actual notebook session.
3. Replace every bracketed prompt with your own analysis or course feedback. Check the dataset provenance and topic labels.
4. Submit the final report and the executed notebook. Keep the result bundle and metrics for reproducibility.